# 中际旭创 300308.SZ 最近一年股价分析

本 Notebook 使用带复权因子的 CSV，默认以前复权价格复现 HTML 页面中的核心指标和图形。

> 数据来源：Tushare daily 与 adj_factor 接口。内容仅用于研究分析，不构成投资建议。


## 复权口径

- 未复权：真实交易价格，适合核对盘面，但遇到分红、送转、拆股时价格序列会跳变。
- 前复权：把历史价格调整到当前口径，最新价格保持真实收盘价，适合技术指标、收益率、回撤和多数回测。
- 后复权：把当前价格调整到历史口径，适合看长期累计财富效应，但最新价格不等于真实盘面价。


In [ ]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_PATH = Path('../data/zhongji_xuchuang_300308_SZ_daily_20250628_20260628.csv')
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
df['date'] = pd.to_datetime(df['trade_date'].astype(str))
df = df.sort_values('date').reset_index(drop=True)
df['ret'] = df['qfq_close'].pct_change().fillna(df['qfq_pct_chg'] / 100)
df.head()


In [ ]:
for window in [5, 10, 20, 60, 120]:
    df[f'MA{window}'] = df['qfq_close'].rolling(window).mean()

df['cummax_close'] = df['qfq_close'].cummax()
df['drawdown'] = df['qfq_close'] / df['cummax_close'] - 1
df['volatility20'] = df['ret'].rolling(20).std() * (252 ** 0.5)
df['volatility60'] = df['ret'].rolling(60).std() * (252 ** 0.5)
summary = pd.Series({
    'start_date': df['date'].iloc[0].date(),
    'end_date': df['date'].iloc[-1].date(),
    'trading_days': len(df),
    'qfq_cum_return_pct': (df['qfq_close'].iloc[-1] / df['qfq_close'].iloc[0] - 1) * 100,
    'raw_cum_return_pct': (df['close'].iloc[-1] / df['close'].iloc[0] - 1) * 100,
    'annualized_vol_pct': df['ret'].std() * (252 ** 0.5) * 100,
    'max_drawdown_pct': df['drawdown'].min() * 100,
    'adj_factor_first': df['adj_factor'].iloc[0],
    'adj_factor_last': df['adj_factor'].iloc[-1],
})
summary


In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.04, row_heights=[0.72, 0.28])
fig.add_trace(go.Candlestick(x=df['date'], open=df['qfq_open'], high=df['qfq_high'], low=df['qfq_low'], close=df['qfq_close'], name='前复权K线', increasing_line_color='#d84a4a', decreasing_line_color='#1f9d78'), row=1, col=1)
for name, color in [('MA5', '#2468d8'), ('MA20', '#b7791f'), ('MA60', '#6b55c5'), ('MA120', '#334155')]:
    fig.add_trace(go.Scatter(x=df['date'], y=df[name], mode='lines', name=name, line=dict(color=color, width=1.6)), row=1, col=1)
fig.add_trace(go.Bar(x=df['date'], y=df['vol'], name='成交量', marker_color=['#d84a4a' if c >= o else '#1f9d78' for c, o in zip(df['qfq_close'], df['qfq_open'])]), row=2, col=1)
fig.update_layout(title='中际旭创 前复权K线、均线与成交量', xaxis_rangeslider_visible=False, height=760, template='plotly_white')
fig.show()


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['date'], y=df['qfq_close'], mode='lines', name='前复权收盘价', line=dict(color='#12314f', width=2.4)))
fig.add_trace(go.Scatter(x=df['date'], y=df['close'], mode='lines', name='未复权收盘价', line=dict(color='#94a3b8', dash='dot')))
fig.add_trace(go.Scatter(x=df['date'], y=df['MA20'], mode='lines', name='MA20', line=dict(color='#b7791f')))
fig.add_trace(go.Scatter(x=df['date'], y=df['MA60'], mode='lines', name='MA60', line=dict(color='#6b55c5')))
fig.update_layout(title='前复权与未复权收盘价对比', yaxis_title='价格', height=520, template='plotly_white')
fig.show()


In [ ]:
fig = make_subplots(rows=2, cols=2, subplot_titles=('成交额', '前复权日收益率分布', '回撤曲线', '滚动年化波动率'))
fig.add_trace(go.Bar(x=df['date'], y=df['amount'], name='成交额', marker_color='#2468d8'), row=1, col=1)
fig.add_trace(go.Histogram(x=df['qfq_pct_chg'], nbinsx=42, name='前复权日收益率', marker_color='#6b55c5'), row=1, col=2)
fig.add_trace(go.Scatter(x=df['date'], y=df['drawdown'] * 100, name='回撤%', fill='tozeroy', line=dict(color='#1f9d78')), row=2, col=1)
fig.add_trace(go.Scatter(x=df['date'], y=df['volatility20'] * 100, name='20日波动率', line=dict(color='#d84a4a')), row=2, col=2)
fig.add_trace(go.Scatter(x=df['date'], y=df['volatility60'] * 100, name='60日波动率', line=dict(color='#2468d8')), row=2, col=2)
fig.update_layout(height=760, template='plotly_white', showlegend=True)
fig.show()
